In [15]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder


df = pd.DataFrame({
    'city': ['Peshawar', 'Lahore', 'Karachi', 'Lahore', 'Peshawar'],
    'education': ['school', 'university', 'college', 'college', 'university'],
    'bought': ['no', 'yes', 'yes', 'no', 'yes']
})


In [6]:
ord_enc = OrdinalEncoder(categories=[['school', 'college', 'university']], dtype='int')
lab_enc = LabelEncoder()
onehot_enc = OneHotEncoder(sparse_output=False, dtype='int', drop='first')

df['edu_enc'] = ord_enc.fit_transform(df[['education']])
df['bought_enc'] = lab_enc.fit_transform(df['bought'])
oh = onehot_enc.fit_transform(df[['city']])
names = onehot_enc.get_feature_names_out()

df[names] = oh
trans_df = df.iloc[:, 3:]
trans_df

,edu_enc,bought_enc,city_Lahore,city_Peshawar
0,0,0,0,1
1,2,1,1,0
2,1,1,0,0
3,1,0,1,0
4,2,1,0,1


# Column Transformer

In the encoding notebook we handled each column on its own. Here is the same work done in one go on the small table:

* `education` is ordinal, so we use `OrdinalEncoder` with the order we choose.
* `bought` is the target, so we use `LabelEncoder`.
* `city` is nominal, so we use `OneHotEncoder` with `drop='first'`.

To do this we had to make three encoders, fit each one on its own column, turn the one hot output into a DataFrame with column names, join it back to `df`, and then slice out the new columns with `df.iloc[:, 3:]`. That works, but it is a lot of manual steps, and it gets worse as the number of columns grows.

There is also a risk. Every step is separate, so it is easy to forget to apply the same steps to the test data later, or to mix up the column order.

`ColumnTransformer` solves this. We tell it which transformer goes to which column, and it applies all of them together in a single step and gives back one combined result.

In [16]:
df

,city,education,bought
0,Peshawar,school,no
1,Lahore,university,yes
2,Karachi,college,yes
3,Lahore,college,no
4,Peshawar,university,yes


In [29]:
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers=[
    ('tnf1', OneHotEncoder(dtype=int, sparse_output=False, drop='first'), ['city']),
    ('tnf2', OrdinalEncoder(dtype=int, categories=[['school', 'college', 'university']]), ['education']),
],
verbose_feature_names_out=False,
remainder='passthrough')

tnf_encoded = ct.fit_transform(df)
tnf_enc_df = pd.DataFrame(tnf_encoded, columns=ct.get_feature_names_out())

In [30]:
tnf_enc_df

,city_Lahore,city_Peshawar,education,bought
0,0,1,0,no
1,1,0,2,yes
2,0,0,1,yes
3,1,0,1,no
4,0,1,2,yes


## Doing It With ColumnTransformer

Now we do the same encoding in one step. Each item in `transformers` is a tuple with three parts:

1. **A name:** any label we choose, like `'tnf1'`. It is only used to identify the step.
2. **The transformer:** what to apply, like `OneHotEncoder` or `OrdinalEncoder`.
3. **The columns:** a list of the columns that transformer works on.

So `('tnf1', OneHotEncoder(...), ['city'])` means: apply one hot encoding to the `city` column. And `('tnf2', OrdinalEncoder(...), ['education'])` means: apply ordinal encoding to `education`, in the order we gave.

Two extra arguments change how the result looks:

* **`remainder='passthrough'`:** any column we did not mention is kept as it is. Here that is `bought`. The default is `'drop'`, which would throw it away.
* **`verbose_feature_names_out=False`:** by default sklearn adds the step name as a prefix, which gives names like `tnf1__city_Lahore`. Setting this to `False` keeps the plain names.

The result is one table: `city` became two columns because of `drop='first'`, `education` became 0, 1, 2, and `bought` stayed as text. We did all of this in a single call, with no joining and no slicing by position.

In [33]:
tnf_enc_df['bought'] = LabelEncoder().fit_transform(tnf_enc_df['bought'])
tnf_enc_df

,city_Lahore,city_Peshawar,education,bought
0,0,1,0,0
1,1,0,2,1
2,0,0,1,1
3,1,0,1,0
4,0,1,2,1


We cannot apply `LabelEncoder` inside `ColumnTransformer`, so we do it separately. `ColumnTransformer` works on input columns and passes each one to its transformer as a table, while `LabelEncoder` is built for a single target column. That is why `bought` goes through `passthrough` and we encode it on its own afterwards.